# Section 5 Next Step - RAG (Retrieval-Augmented Generation)

The storage step built an index. This notebook adds the final layer: **ask a question, retrieve the relevant chunks, and let an LLM answer using only your content**.

---
## The Two Lanes

Everything in this course runs in one of two directions:

| Lane | Steps | Runs | Purpose |
|------|-------|------|---------|
| **Offline** (one-time) | ingest -> split -> embed -> store | on build | build the catalog |
| **Online** (per question) | embed question -> retrieve -> prompt -> LLM | every query | answer using the catalog |

You built the offline lane in `section_5_master`. RAG is the **online** lane that consumes it: we reuse the index and append the "ask and answer" loop.

---
## The Three Building Blocks of RAG

1. **Retriever** - wraps the vector store and fetches the `k` most similar chunks for a query.
2. **PromptTemplate** - a text recipe with `{context}` and `{question}` slots.
3. **ChatLLM** - a *generative* model (Ollama, `llama3:8b`) that reads the filled prompt and writes a grounded answer.

These are joined by **LCEL** (LangChain Expression Language) - the `|` pipe syntax that composes them into a single `chain`.

---
## Setup

We rebuild the album index in-memory (the same one from the master notebook), then expose it as a **retriever**.

In [ ]:

from pathlib import Path
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

def find_data_dir():
    for root in (Path.cwd(), Path.cwd().parent, Path.cwd().parents[1]):
        cand = root / "data_ingestion"
        if cand.is_dir():
            return cand
    raise FileNotFoundError("Could not find data_ingestion folder")

DATA = find_data_dir()

# --- same splitter + embeddings as the master notebook ---
splitter = RecursiveCharacterTextSplitter(chunk_size=120, chunk_overlap=20)
emb      = OllamaEmbeddings(model="nomic-embed-text")

# --- rebuild the album index (self-contained so this notebook runs alone) ---
album_docs = TextLoader(str(DATA / "heylog_eve_album.txt")).load()
album_chunks = splitter.split_documents(album_docs)
store = FAISS.from_documents(album_chunks, emb)
print("album index chunks:", store.index.ntotal)

---
## 1. Retriever

A retriever is a thin wrapper over the vector store that offers an `invoke(query)` method. It:
- embeds the query (via the same `embed_query` path)
- runs a similarity search
- returns the top-`k` chunk `Document`s

The retriever hides the store internals, so any vector store (FAISS, Chroma, Pinecone...) can be swapped in later.

In [ ]:

# wrap the store as a retriever, asking for the top 3 matches per query
retriever = store.as_retriever(search_kwargs={"k": 3})

# invoke returns the k closest chunks as Document objects
hits = retriever.invoke("shame and hiding like Adam and Eve")

print(f"retriever returned {len(hits)} chunks")
for i, d in enumerate(hits):
    print(f"\n[hit {i}]")
    print("  source:", d.metadata["source"].split("/")[-1])
    print("  text:", d.page_content[:80].replace(chr(10), " "))

## Formatting the context

The retriever returns a list of `Document`s, but the prompt needs a single string. We join all chunk texts into one block separated by blank lines.

In [ ]:

# helper applied to the retriever output: list[Document] -> one string
def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

# demo on the hits we already retrieved
context_block = format_docs(hits)
print("context_block is a:", type(context_block).__name__)
print("length:", len(context_block), "chars")
print("--- start ---")
print(context_block[:200])

---
## 2. Prompt Template

A `PromptTemplate` is a string with `{placeholders}`. At run time we fill them in. For RAG we use two slots:
- `{context}` - the retrieved, formatted chunks
- `{question}` - the user's question

The instructions tell the LLM to **answer only from the context**, which is what makes the answer grounded in your data.

In [ ]:
# build the prompt template once; it is reused for every question
prompt = PromptTemplate.from_template(
    "You are a helpful assistant. Answer using ONLY the context provided.\n"
    "If the context does not contain the answer, say \"I don't know.\"\n"
    "\n"
    "Context:\n"
    "{context}\n"
    "\n"
    "Question: {question}\n"
    "Answer:"
)

# show how the prompt fills in (no LLM yet, just string formatting)
filled = prompt.format(context="[chunk 1]\n[chunk 2]", question="what is it?")
print(filled)

---
## 3. Chat LLM (Ollama)

The embedding model (`nomic-embed-text`) turns text into **vectors** - it cannot write answers. For generation we need a **chat/generative** model. You already have several pulled locally:

- `llama3:8b` - default, best quality (slower)
- `phi3:mini` - small and fast
- `mistral:7b` - balanced

`ChatOllama` returns an `AIMessage` object; we unwrap `.content` or pipe through `StrOutputParser`.

In [ ]:

# load the generative model from Ollama
llm = ChatOllama(model="llama3:8b")   # swap to "phi3:mini" for speed

# direct call returns an AIMessage
msg = llm.invoke("Reply with exactly: hello world")
print("type:", type(msg).__name__)
print("content:", msg.content)

---
## The Full RAG Chain (LCEL)

LCEL lets us compose blocks with `|`. The tricky part is that the retriever needs the query, and the prompt needs `{context}` + `{question}`. We solve this with a small dict that splays the question into two jobs:

- `{"context": retriever | format_docs, ...}` - for a given question, run the retriever then format the chunks -> becomes `context`
- `{"question": RunnablePassthrough()}` - pass the raw question through unchanged -> becomes `question`

So the chain dataflow is:
`question -> (retrieve+format) && (passthrough) -> prompt -> llm -> parser`

In [ ]:

# end-to-end chain composition using LCEL
chain = (
    {
        "context":  retriever | format_docs,      # retrieve + join into one string
        "question": RunnablePassthrough(),        # keep the raw question as-is
    }
    | prompt                                     # fill the {context}/{question} slots
    | llm                                        # generate the grounded answer
    | StrOutputParser()                          # unwrap AIMessage -> plain str
)

# ask a question grounded in the album content
answer = chain.invoke("what is the song paranoid about?")
print("ANSWER:")
print(answer)

---
## Inspecting What the Model Saw

A powerful debugging habit: run the same query through the retriever alone to see exactly which chunks grounded the answer. This gives you **provenance** - you can verify the model didn't hallucinate.

In [ ]:

query = "what is the song paranoid about?"

# what the chain actually fed the LLM (retrieved chunks)
grounding = retriever.invoke(query)
print("=== chunks retrieved for: '{query}' ===")
for i, d in enumerate(grounding):
    print(f"[chunk {i}] {d.page_content[:75].replace(chr(10),' ')}")

# contrast: a query with no matching content should hit the "I don't know" guard
answer2 = chain.invoke("what color are the carpets in the album?")
print("\n=== answer to out-of-scope query ===")
print(answer2)

---
## Summary

RAG closes the loop on this section's pipeline:

- **Offline** (already built): `ingest -> split -> embed -> store`
- **Online (this notebook)**: `retriever -> prompt -> LLM -> answer`

The three blocks + LCEL glue:
```python
chain = {"context": retriever | format_docs, "question": RunnablePassthrough()}\
         | prompt | llm | StrOutputParser()
```

**Key takeaways**
- Retrievers abstract the vector store; swap backends freely.
- Embedding models find; chat models generate - never confuse the two.
- Prompts with `{context}` slots make answers **grounded** in your content.
- LCEL `|` composition is the modern (Runnable) way to build chains.

**Where this goes next (logically):**
1. **Memory / conversation** - carry prior turns so follow-ups make sense.
2. **Tools** - let the model request function calls (including this retriever).
3. **Agents** - the model loops and decides which tools to call.
4. **LangGraph** - the modern framework for stateful agent graphs.

This makes RAG the bridge from pure retrieval into the **Agentic** half of this repo.